In [73]:
from lokigi.travel_utils import prepare_valhalla_network, build_time_matrix_valhalla
import geopandas
import pandas as pd
import datetime
from pyrosm import OSM

In [74]:
prepare_valhalla_network(
    osm_path="devon-260422-modified-traffic.osm.pbf",
    output_dir="valhalla/",
    output_name="devon-260422-modified-traffic-valhalla"
)

Using existing Valhalla build


{'config_path': 'C:\\geographic_or_ds_playground\\data\\travel_matrix_generation\\valhalla\\devon-260422-modified-traffic-valhalla.json',
 'tile_dir': 'C:\\geographic_or_ds_playground\\data\\travel_matrix_generation\\valhalla\\devon-260422-modified-traffic-valhalla_tiles',
 'traffic_path': 'C:\\geographic_or_ds_playground\\data\\travel_matrix_generation\\valhalla\\devon-260422-modified-traffic-valhalla_traffic.tar'}

We then manually modify the maximum radius in the config file so that the router doesn't 'give up' too easily. 

This feature will be available in future versions of lokigi by default. 

As this won't rebuild if the file is already present, we can safely rerun this cell. 

## Set up centroids

In [75]:
pw_centroids_gdf = geopandas.read_file("LSOA_PopCentroids_EW_2021_V4_-4541397882496207062.gpkg")

### Limit centroid dataset to Devon

In [76]:
devon_lsoas = geopandas.read_file("../LSOA_Devon_2021_EW_BSC_V4.gpkg")

In [77]:
origins_gdf = pw_centroids_gdf.merge(devon_lsoas[["LSOA21CD","LSOA21NM"]], on="LSOA21CD", how="right")

In [78]:
origins_gdf

,LSOA21CD,GlobalID,GlobalID_2,geometry,LSOA21NM
0,E01015023,0d153926-4cb8-4c73-936f-14bb7f8400e9,{73746429-2FEB-4369-9599-4521D49E11EB},POINT (248037.784 59438.321),Plymouth 003A
1,E01015024,114f181e-e872-41e2-b62e-2dc890416901,{823B086A-4A23-4163-ADF5-E48A1D382A12},POINT (246376.097 60366.097),Plymouth 004A
2,E01015025,b8300906-e24e-448e-b9ad-3d13d5b8a422,{213C532D-0547-4921-BBFF-0AB449558887},POINT (249312.961 60296.177),Plymouth 001A
3,E01015026,93c6b322-fcc3-4a1c-9241-637d155f5a80,{C135377D-78CD-4D60-8A32-C25546E58CAB},POINT (246654.399 60075.381),Plymouth 003B
4,E01015027,ac286bc6-4836-4375-884b-5377fe5e8e00,{60A29F98-5ABD-4FDD-B24F-7CF3CDBEEBB5},POINT (248567.884 59967.869),Plymouth 002A
...,...,...,...,...,...
724,E01034639,700ebf4f-7441-4572-93bf-1fc255250cbe,{1AD9F15B-4BDA-47FB-A4EA-9F77F6DE687C},POINT (253128.556 132278.289),North Devon 012G
725,E01034640,78b9d8eb-394d-4b5d-b3fb-b1d05dcc3b63,{A6E2E170-14CB-4F35-838E-6E3F0CA2E5B6},POINT (243407.044 129119.951),Torridge 002D
726,E01034641,99364620-e9d1-4447-9ed3-79edfbe2084d,{82A971A5-4C7F-4FA8-90FE-36D7926B4D4C},POINT (243076.758 128716.969),Torridge 002E
727,E01034642,ac365516-ca2c-4d20-8edd-bd1f4389121c,{0C677362-5DFB-44D7-8A9C-0AE60BC26095},POINT (250565.417 98951.176),West Devon 001E


### Snap to nearest navigable road 

This avoids weird quirks where the PWC is in the middle of a very rural area like Dartmoor and a long walking time gets added on. 

In [79]:
osm = OSM("devon-260422-modified-traffic.osm.pbf")

In [80]:
roads = osm.get_network(network_type="driving")

In [81]:
roads = roads.to_crs("EPSG:27700")

In [82]:
roads_sindex = roads.sindex
roads_sindex

In [83]:
nearest_roads = origins_gdf.sjoin_nearest(
    roads[["geometry"]],
    how="left",
    distance_col="distance_to_road"
)

In [84]:
nearest_roads.sort_values('distance_to_road', ascending=False)

,LSOA21CD,GlobalID,GlobalID_2,geometry,LSOA21NM,index_right,distance_to_road
517,E01020174,5c2438d4-54c3-498a-b470-f1eb94a8a6e3,{6C0F46B4-C863-4C64-8813-3305C5D7EB07},POINT (276521.067 42150.957),South Hams 011A,27301,500.701549
627,E01020286,97023a64-d0a2-4549-9f15-6eeed775d1da,{C74A1501-2420-484A-A7F5-07F7B09C580A},POINT (236708.847 91328.02),Torridge 009A,79906,398.963027
615,E01020274,6b043f08-7f78-4d88-b85a-39b538d2a370,{549BD68E-85E9-4B3E-8183-345C29808BD3},POINT (284349.712 86211.185),Teignbridge 001D,79332,356.682663
578,E01020237,acdec106-4acb-4717-950f-fda3e5b8e885,{91323D26-6840-4687-B1D0-7DE170744F37},POINT (279497.826 74740.568),Teignbridge 007D,97330,352.434254
587,E01020246,1de491cf-7c88-4d3c-822f-9cd19bc3022e,{495C00E1-CB99-4937-BD66-BB67BABADD99},POINT (288977.839 70412.144),Teignbridge 018A,23919,331.309757
...,...,...,...,...,...,...,...
110,E01015133,aa95fd50-27f1-406a-87f0-1a9209ee9a54,{33734A4F-DEF0-4BA9-90E7-DD4114C544F3},POINT (252592.788 53314.717),Plymouth 030E,6045,0.173427
391,E01020046,cc877964-fcd9-4d0a-a8cc-9af265bff159,{BE7991BF-D525-43CF-98E0-8EE4C3D179C9},POINT (305661.462 118354.786),Mid Devon 001A,94982,0.141625
698,E01034168,d0c49b5e-4111-4a5c-b030-381d19616f2b,{7724102C-551B-4565-8DD1-772FCF54DFC6},POINT (283435.22 72189.51),Teignbridge 013F,70974,0.100536
173,E01015201,c77730d6-1b08-40d9-96c9-2e7cf3f50239,{A765E0AA-2FF1-42E4-AF8D-83126960F5F9},POINT (288894.69 56277.402),Torbay 016E,2271,0.096400


In [85]:
nearest_roads["road_geometry"] = (
    roads.loc[nearest_roads["index_right"], "geometry"]
    .values
)

nearest_roads["snapped_geometry"] = nearest_roads.apply(
    lambda row: row["road_geometry"].interpolate(
        row["road_geometry"].project(row.geometry)
    ),
    axis=1,
)

nearest_roads

,LSOA21CD,GlobalID,GlobalID_2,geometry,LSOA21NM,index_right,distance_to_road,road_geometry,snapped_geometry
0,E01015023,0d153926-4cb8-4c73-936f-14bb7f8400e9,{73746429-2FEB-4369-9599-4521D49E11EB},POINT (248037.784 59438.321),Plymouth 003A,110505,16.836545,"MULTILINESTRING ((248070.593 59403.473, 248022...",POINT (248030.368 59423.206)
1,E01015024,114f181e-e872-41e2-b62e-2dc890416901,{823B086A-4A23-4163-ADF5-E48A1D382A12},POINT (246376.097 60366.097),Plymouth 004A,11664,3.160587,"MULTILINESTRING ((246316.073 60302.756, 246317...",POINT (246377.719 60363.384)
2,E01015025,b8300906-e24e-448e-b9ad-3d13d5b8a422,{213C532D-0547-4921-BBFF-0AB449558887},POINT (249312.961 60296.177),Plymouth 001A,111380,18.169709,"MULTILINESTRING ((249226.389 60273.234, 249253...",POINT (249314.304 60278.057)
3,E01015026,93c6b322-fcc3-4a1c-9241-637d155f5a80,{C135377D-78CD-4D60-8A32-C25546E58CAB},POINT (246654.399 60075.381),Plymouth 003B,104062,1.333731,"MULTILINESTRING ((246635.957 60088.598, 246644...",POINT (246654.95 60076.595)
4,E01015027,ac286bc6-4836-4375-884b-5377fe5e8e00,{60A29F98-5ABD-4FDD-B24F-7CF3CDBEEBB5},POINT (248567.884 59967.869),Plymouth 002A,111375,11.684899,"MULTILINESTRING ((248649.763 59945.772, 248614...",POINT (248571.711 59978.91)
...,...,...,...,...,...,...,...,...,...
724,E01034639,700ebf4f-7441-4572-93bf-1fc255250cbe,{1AD9F15B-4BDA-47FB-A4EA-9F77F6DE687C},POINT (253128.556 132278.289),North Devon 012G,22625,0.228774,"MULTILINESTRING ((253127.472 132533.117, 25311...",POINT (253128.78 132278.335)
725,E01034640,78b9d8eb-394d-4b5d-b3fb-b1d05dcc3b63,{A6E2E170-14CB-4F35-838E-6E3F0CA2E5B6},POINT (243407.044 129119.951),Torridge 002D,31655,7.322407,"MULTILINESTRING ((243364.539 129109.301, 24338...",POINT (243403.834 129126.533)
726,E01034641,99364620-e9d1-4447-9ed3-79edfbe2084d,{82A971A5-4C7F-4FA8-90FE-36D7926B4D4C},POINT (243076.758 128716.969),Torridge 002E,27896,11.810475,"MULTILINESTRING ((243255.029 128815.609, 24324...",POINT (243071.704 128727.644)
727,E01034642,ac365516-ca2c-4d20-8edd-bd1f4389121c,{0C677362-5DFB-44D7-8A9C-0AE60BC26095},POINT (250565.417 98951.176),West Devon 001E,77169,86.336724,"MULTILINESTRING ((250719.162 99068.493, 250727...",POINT (250574.324 98865.3)


In [86]:
origins_gdf = nearest_roads.copy()

origins_gdf = origins_gdf.drop(
    columns=["road_geometry", "geometry"]
).rename(columns={"snapped_geometry": "geometry"})

origins_gdf = origins_gdf.set_geometry(
    "geometry"
)


In [87]:
origins_gdf = origins_gdf.set_crs("EPSG:27700")

In [88]:
origins_gdf = origins_gdf.to_crs("EPSG:4326").drop_duplicates("LSOA21NM")

In [89]:
origins_gdf.to_file("snapped_pwc.gpkg", driver="gpkg")

## Set up destinations

In [90]:
destinations_df = pd.read_csv("../../data/devon_cdcs.csv")

destinations_gdf = geopandas.GeoDataFrame(
        destinations_df,  # Our pandas dataframe
        geometry=geopandas.points_from_xy(
            destinations_df[
                "Longitude"
            ],  # Our 'x' column (horizontal position of points)
            destinations_df["Latitude"],  # Our 'y' column (vertical position of points)
        ),
        crs="EPSG:4326",
    )

destinations_gdf

,Facility_Name,Latitude,Longitude,Existing,geometry
0,Bideford Community Hospital,51.018099,-4.212523,Yes,POINT (-4.21252 51.0181)
1,NHS Nightingale Exeter,50.723418,-3.466705,Yes,POINT (-3.4667 50.72342)
2,"Colin Campbell Court, Plymouth",50.370673,-4.148157,Yes,POINT (-4.14816 50.37067)
3,Torbay & South Devon Community Diagnostic Centre,50.466924,-3.528502,Yes,POINT (-3.5285 50.46692)
4,Tiverton - Lowman Way,50.911822,-3.465859,No,POINT (-3.46586 50.91182)
5,Okehampton - Exeter Road Industrial Estate,50.742219,-3.980639,No,POINT (-3.98064 50.74222)
6,Crediton - Lords Meadow,50.791319,-3.638029,No,POINT (-3.63803 50.79132)
7,South Molton - Pathfields,51.027692,-3.832400,No,POINT (-3.8324 51.02769)
8,Newton Abbott - Market Walk,50.530644,-3.611170,No,POINT (-3.61117 50.53064)
9,Bovey Tracey - Blue Waters,50.581513,-3.673217,No,POINT (-3.67322 50.58151)


In [91]:
destinations_gdf.explore()

In [92]:
origins_gdf_final=origins_gdf.rename(columns={'LSOA21NM':'id'})[['id','geometry']]
origins_gdf_final

,id,geometry
0,Plymouth 003A,POINT (-4.14035 50.41497)
1,Plymouth 004A,POINT (-4.16398 50.42298)
2,Plymouth 001A,POINT (-4.12263 50.42298)
3,Plymouth 003B,POINT (-4.15996 50.42048)
4,Plymouth 002A,POINT (-4.13296 50.4201)
...,...,...
724,North Devon 012G,POINT (-4.0977 51.07099)
725,Torridge 002D,POINT (-4.23504 51.0401)
726,Torridge 002E,POINT (-4.2396 51.03642)
727,West Devon 001E,POINT (-4.12044 50.77008)


In [93]:
destinations_gdf_final = destinations_gdf.rename(columns={'Facility_Name':'id'})[['id','geometry']]
destinations_gdf_final

,id,geometry
0,Bideford Community Hospital,POINT (-4.21252 51.0181)
1,NHS Nightingale Exeter,POINT (-3.4667 50.72342)
2,"Colin Campbell Court, Plymouth",POINT (-4.14816 50.37067)
3,Torbay & South Devon Community Diagnostic Centre,POINT (-3.5285 50.46692)
4,Tiverton - Lowman Way,POINT (-3.46586 50.91182)
5,Okehampton - Exeter Road Industrial Estate,POINT (-3.98064 50.74222)
6,Crediton - Lords Meadow,POINT (-3.63803 50.79132)
7,South Molton - Pathfields,POINT (-3.8324 51.02769)
8,Newton Abbott - Market Walk,POINT (-3.61117 50.53064)
9,Bovey Tracey - Blue Waters,POINT (-3.67322 50.58151)


In [94]:
travel_time_matrix_car_valhalla = build_time_matrix_valhalla(
    origins_gdf=origins_gdf_final,
    destinations_gdf=destinations_gdf_final,
    valhalla_config_path="valhalla/devon-260422-modified-traffic-valhalla.json",
    output_csv_path="../travel_matrix_car.csv",
    costing="auto", # This means car, not 'automatic'!
    reshape_to_wide=True
)

Streaming Wide Matrix Rows: 100%|██████████| 15/15 [00:33<00:00,  2.24s/it]

Matrix generation complete. Output saved to ../travel_matrix_car.csv


## Use r5py for public transport

In [22]:
import r5py

transport_network = r5py.TransportNetwork(
    "devon-260422-modified.osm.pbf",
    [
        "itm_south_west_gtfs.zip",
    ]
)

In [23]:
travel_time_matrix_transit = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins_gdf_final,
    destinations=destinations_gdf_final,
    transport_modes=[r5py.TransportMode.TRANSIT],
    departure=datetime.datetime(2026, 4, 23, 14, 0, 0),
    max_time=datetime.timedelta(minutes=600),
    departure_time_window=datetime.timedelta(minutes=60),
    max_public_transport_rides=20
)

In [24]:
travel_time_matrix_transit_wide = travel_time_matrix_transit.pivot(columns="to_id", index="from_id", values="travel_time").reset_index()

In [25]:
travel_time_matrix_transit_wide_extended = travel_time_matrix_transit_wide.set_index("from_id").dropna(how="all").reset_index(drop=False)


In [26]:
travel_time_matrix_transit_wide_extended = travel_time_matrix_transit_wide_extended.fillna(9999.0)

In [27]:
travel_time_matrix_transit_wide_extended.to_csv("../travel_matrix_public_transport.csv", index=False)